# 🔍 Enterprise AI Security RAG & Observability Lab with LangSmith

---

> **Learning-only note:** This notebook uses synthetic policies, dummy credentials, and fictional operational values. It is a hands-on observability lab, not a real company security standard or production system.


## What We Are Building & Observing:
1. **Tracing Fundamentals**: Custom `@traceable` spans (`run_type="retriever"`, `run_type="llm"`, `run_type="chain"`).
2. **Production Security RAG Pipeline**: Indexing `enterprise_ai_security_policy.pdf` with `PyPDFLoader` $\rightarrow$ `RecursiveCharacterTextSplitter` $\rightarrow$ `OpenAIEmbeddings` $\rightarrow$ `InMemoryVectorStore` $\rightarrow$ `ChatPromptTemplate` $\rightarrow$ `ChatOpenAI`.
3. **LangSmith Observability & Telemetry**: Attaching custom run names, tags, and metadata to live traces.
4. **Programmatic APM**: Inspecting run latency, token costs, and status via the LangSmith `Client`.
5. **Automated Evaluations**: Building a golden test dataset and running automated evaluators to score policy accuracy.


In [1]:
# 1. Setup Environment Variables and LangSmith Tracing
import os
import time
from dotenv import load_dotenv
from langsmith import traceable

# Load keys from .env file
load_dotenv(override=True)

# LangSmith Tracing Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "").strip('"')
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "ai-security-observability-lab").strip('"')

print("✅ LangSmith Tracing configured for project:", os.environ.get("LANGCHAIN_PROJECT"))
print("✅ OpenAI API Key set:", bool(os.environ.get("OPENAI_API_KEY")))
print("✅ LangSmith API Key set:", bool(os.environ.get("LANGCHAIN_API_KEY")))

# Tracing Fundamentals: Defining Spans
@traceable(run_type="retriever", name="Mock Document Retriever")
def search_docs(question):
    time.sleep(0.2)
    return ["refund_policy.md", "faq.md"]

@traceable(run_type="llm", name="Mock LLM Generator")
def LLM_Fake(question, docs):
    time.sleep(0.4)
    return "Based on " + str(len(docs)) + " documents, here is the answer."

@traceable(run_type="chain", name="Mock Pipeline")
def pipeline(question):
    docs = search_docs(question)
    answer = LLM_Fake(question, docs)
    return answer

print("Result from Mock Pipeline:", pipeline("what is refund policy?"))


✅ LangSmith Tracing configured for project: observability
✅ OpenAI API Key set: True
✅ LangSmith API Key set: True


Result from Mock Pipeline: Based on 2 documents, here is the answer.


In [2]:
# 2. Verify OpenAI Model Connectivity
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
reply = llm.invoke("reply with exactly: langsmith is listening")
print("LLM Response:", reply.content)


LLM Response: langsmith is listening


In [3]:
# 3. PDF Ingestion & Text Splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("enterprise_ai_security_policy.pdf")
Original_docs = loader.load()
print(f"✅ Loaded PDF: {len(Original_docs)} pages")

text_Splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
docs = text_Splitter.split_documents(Original_docs)
print(f"✅ Split into {len(docs)} chunks")
print("--- Sample Chunk 1 Preview ---")
print(docs[0].page_content)
print("Metadata:", docs[0].metadata)


✅ Loaded PDF: 2 pages
✅ Split into 13 chunks
--- Sample Chunk 1 Preview ---
■■ Enterprise AI Security & Incident Response
Standard
Document Version: 3.2.0 | Classification: CONFIDENTIAL - INTERNAL USE ONLY | Effective Date: 2026-Q1
1. Purpose, Scope & Governance
This document establishes the mandatory security controls, operational guardrails, and compliance standards
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T19:44:46-04:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T19:44:46-04:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'enterprise_ai_security_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}


In [4]:
# 4. Embeddings, Vector Store & Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(docs, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Test the retriever
retrieved_docs = retriever.invoke("What is the SLA for isolating a breached model endpoint?")
for i, doc in enumerate(retrieved_docs):
    print(f"--- RETRIEVED CHUNK {i+1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- RETRIEVED CHUNK 1 ---
5. High-Severity Incident Response & Emergency Containment (IRP)
 Protocol IR-401 (Isolation SLA): In the event of a confirmed model exploit, data exfiltration breach, or autonomous
agent loop runaway, the SecOps team must isolate the model endpoint within 5 minutes.
 Protocol IR-402 (SOC Escalation): Security Operations Center (SOC) must be automatically paged via PagerDuty
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T19:44:46-04:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T19:44:46-04:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'enterprise_ai_security_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
--- RETRIEVED CHUNK 2 ---
instructions.
 Rule PI-203 (Canary Tokens): System prompts must embed unique cryptographic canary tokens. If a model output
contains the canary token, the session must be immediately

In [5]:
# 5. Prompt Template & LCEL Chain Assembly
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Helper function to format retrieved documents
def format_docs(retrieved_documents):
    return "\n\n".join(doc.page_content for doc in retrieved_documents)

# Security prompt template with strict citation rules
prompt_template = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert Enterprise AI Security Analyst.\n"
        "Answer the question using ONLY the provided security context below.\n"
        "If the information is not in the context, say: 'I cannot find this in the policy.'\n"
        "Always cite the exact rule code (e.g., Rule AC-102, Protocol IR-401, Rule OB-502)."
    )),
    ("human", "Security Context:\n{context}\n\nQuestion: {question}")
])

# Assemble the complete LCEL RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("🚀 Complete Enterprise Security RAG Chain is ready with automatic LangSmith tracing!")


🚀 Complete Enterprise Security RAG Chain is ready with automatic LangSmith tracing!


In [6]:
# 6. Execute RAG Query with LangSmith Metadata & Tags
question = "What is the mandatory API key rotation schedule, and what is the SLA for isolating a breached model endpoint?"

# Invoke the chain with LangSmith observability configuration
response = rag_chain.invoke(
    question,
    config={
        "run_name": "enterprise_security_rag_lookup",
        "tags": ["rag", "security-compliance", "v3.2.0"],
        "metadata": {
            "environment": "production",
            "source_doc": "enterprise_ai_security_policy.pdf",
            "department": "SecOps",
            "query_type": "incident_and_key_rotation"
        }
    }
)

print("=" * 60)
print("QUESTION:", question)
print("-" * 60)
print("ANSWER:")
print(response)
print("=" * 60)
print(f"👉 Trace sent to LangSmith project: '{os.environ.get('LANGCHAIN_PROJECT')}'")


QUESTION: What is the mandatory API key rotation schedule, and what is the SLA for isolating a breached model endpoint?
------------------------------------------------------------
ANSWER:
The mandatory API key rotation schedule is every 30 calendar days as per Rule AC-102. The SLA for isolating a breached model endpoint is within 15 minutes of breach detection as per Rule AC-103.
👉 Trace sent to LangSmith project: 'observability'


In [7]:
# 7. Programmatic APM: Inspect Runs & Latency via LangSmith Client
from langsmith import Client

client = Client()

runs = list(client.list_runs(
    project_name=os.environ.get("LANGCHAIN_PROJECT", "ai-security-observability-lab"),
    limit=5
))

print(f"📊 Found {len(runs)} recent trace runs in project '{os.environ.get('LANGCHAIN_PROJECT')}':\n")
for r in runs:
    latency_ms = (r.end_time - r.start_time).total_seconds() * 1000 if (r.end_time and r.start_time) else 0
    status_icon = "🟢" if r.status == "success" else "🔴"
    print(f"{status_icon} Run Name: {r.name}")
    print(f"   Run Type: {r.run_type} | ID: {r.id}")
    print(f"   Status: {r.status} | Latency: {latency_ms:.1f} ms | Tokens: {r.total_tokens}")
    print("-" * 50)


📊 Found 5 recent trace runs in project 'observability':

🟢 Run Name: ChatOpenAI
   Run Type: llm | ID: 01a05ff5-3c53-75b3-a210-4686906c7143
   Status: success | Latency: 853.8 ms | Tokens: 19
--------------------------------------------------
🟢 Run Name: Mock LLM Generator
   Run Type: llm | ID: 0b09305d-aa03-40f3-89cd-0829d8327f1f
   Status: success | Latency: 404.7 ms | Tokens: 0
--------------------------------------------------
🟢 Run Name: Mock Document Retriever
   Run Type: retriever | ID: ccb52977-124f-429f-9b4d-e10ae5b8543e
   Status: success | Latency: 204.1 ms | Tokens: 0
--------------------------------------------------
🟢 Run Name: Mock Pipeline
   Run Type: chain | ID: c12c4525-b57e-4cbc-bdd9-dd5ea22f8b82
   Status: success | Latency: 622.7 ms | Tokens: 0
--------------------------------------------------
🟢 Run Name: Incident Severity Calculator
   Run Type: tool | ID: ed005f47-64d4-4f3c-9349-484a3ff4d8c9
   Status: success | Latency: 0.3 ms | Tokens: 0
-------------------

In [8]:
# 9. Two Small Practical Functions with @traceable
from langsmith import traceable

# Function 1: Small Tool to Mask API Keys (run_type="tool")
@traceable(run_type="tool", name="mask_api_key")
def mask_api_key(api_key: str) -> str:
    """Masks sensitive API key to prevent secret leakage."""
    if len(api_key) > 8:
        return api_key[:4] + "..." + api_key[-4:]
    return "[REDACTED]"

# Function 2: Small Chain using the Tool + Retriever (run_type="chain")
@traceable(run_type="chain", name="quick_policy_check")
def quick_policy_check(question: str, user_api_key: str) -> dict:
    """Performs a fast security check using our tool and real PDF retriever."""
    safe_key = mask_api_key(user_api_key)
    matching_docs = retriever.invoke(question)
    
    return {
        "masked_key": safe_key,
        "question": question,
        "matched_policy_rule": matching_docs[0].page_content[:120] + "..."
    }

# Run the 2 small functions
result = quick_policy_check(
    question="What is the key rotation policy?", 
    user_api_key="sk-dummy-learning-0000abcdef"  # DUMMY credential for this lab only
)

print("=" * 60)
print("✅ Result from quick_policy_check:")
print("Masked Key:", result["masked_key"])
print("Question:", result["question"])
print("Matched Policy:", result["matched_policy_rule"])
print("=" * 60)
print("👉 Open LangSmith to see the 2-span trace: quick_policy_check -> mask_api_key!")


✅ Result from quick_policy_check:
Masked Key: sk-p...cdef
Question: What is the key rotation policy?
Matched Policy: endpoints, vector databases, and LangSmith observability dashboards.
 Rule AC-102 (Key Rotation): All AI API keys (Open...
👉 Open LangSmith to see the 2-span trace: quick_policy_check -> mask_api_key!


---

# 10. Second Observability Experiment: IT Helpdesk RAG (Synthetic)

This section keeps the original enterprise-security lab above and adds a simpler **IT Helpdesk RAG** scenario so the traces are easier to explain visually.

### Learning goal
Use LangSmith to compare what happens when:
1. the knowledge base clearly contains the answer,
2. the question is ambiguous, and
3. the answer does not exist in the knowledge base.

**All policies below are fictional and created only for learning.**

In [ ]:
# 10.1 Create a Small Synthetic IT Helpdesk Knowledge Base
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langsmith import traceable

# Reuse the environment/tracing configuration from the earlier cells.
# These policies are intentionally fictional so the experiment is safe to share.
helpdesk_docs = [
    Document(
        page_content=(
            "Policy HD-101 (Lost or Stolen Device): Employees must report a lost or stolen "
            "company laptop to the IT Service Desk within 30 minutes of discovery. IT will "
            "remotely lock the device and start the incident-response process."
        ),
        metadata={"policy_id": "HD-101", "topic": "lost_device", "synthetic": True},
    ),
    Document(
        page_content=(
            "Policy HD-102 (VPN Troubleshooting): If VPN access fails, first confirm internet "
            "connectivity, restart the VPN client, and complete MFA again. If the problem continues, "
            "open a Service Desk ticket and include the error message."
        ),
        metadata={"policy_id": "HD-102", "topic": "vpn", "synthetic": True},
    ),
    Document(
        page_content=(
            "Policy HD-103 (Password Reset): Employees should use the self-service password reset "
            "portal. After five failed sign-in attempts, the account is locked for 15 minutes. "
            "Contact the Service Desk if self-service reset is unavailable."
        ),
        metadata={"policy_id": "HD-103", "topic": "password", "synthetic": True},
    ),
    Document(
        page_content=(
            "Policy HD-104 (Software Installation): Applications in the approved software catalog "
            "may be installed without additional approval. Software outside the catalog requires "
            "manager approval and an IT security review."
        ),
        metadata={"policy_id": "HD-104", "topic": "software", "synthetic": True},
    ),
    Document(
        page_content=(
            "Policy HD-105 (Laptop Replacement): A damaged company laptop must be assessed by IT. "
            "If replacement is approved, a standard replacement device is targeted within two business days."
        ),
        metadata={"policy_id": "HD-105", "topic": "replacement", "synthetic": True},
    ),
    Document(
        page_content=(
            "Policy HD-106 (Account Lockout): Employees can use the self-service unlock flow after "
            "identity verification. Repeated lockouts should be escalated to the Service Desk for investigation."
        ),
        metadata={"policy_id": "HD-106", "topic": "lockout", "synthetic": True},
    ),
]

# Reuse earlier model objects if available; otherwise create them.
if "embeddings" not in globals():
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
if "llm" not in globals():
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"✅ Created {len(helpdesk_docs)} synthetic helpdesk policy documents")

In [ ]:
# 10.2 Build the Helpdesk Vector Store
helpdesk_vector_store = InMemoryVectorStore.from_documents(
    helpdesk_docs,
    embedding=embeddings,
)

print("✅ Helpdesk vector store is ready")

In [ ]:
# 10.3 Trace Retrieval and Expose Similarity Scores
@traceable(run_type="retriever", name="retrieve_it_policy")
def retrieve_it_policy(question: str, k: int = 2) -> list[dict]:
    """Retrieve the most relevant synthetic IT policies and keep their similarity scores visible."""
    results = helpdesk_vector_store.similarity_search_with_score(question, k=k)

    return [
        {
            "content": doc.page_content,
            "metadata": doc.metadata,
            "similarity_score": round(float(score), 4),
        }
        for doc, score in results
    ]

# Quick retrieval-only check
retrieval_preview = retrieve_it_policy("My company laptop was stolen. What should I do?")
for item in retrieval_preview:
    print(item["metadata"]["policy_id"], "score:", item["similarity_score"])
    print(item["content"])
    print("-" * 70)

In [ ]:
# 10.4 Trace the Full IT Helpdesk RAG Flow
from langchain_core.prompts import ChatPromptTemplate

helpdesk_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an IT Helpdesk assistant for a synthetic learning lab. "
        "Answer ONLY from the provided policy context. "
        "If the context does not contain the answer, say exactly: "
        "'I couldn't find this in the IT policy.' "
        "When possible, cite the policy ID."
    ),
    ("human", "Policy Context:\n{context}\n\nQuestion: {question}"),
])

@traceable(run_type="chain", name="it_helpdesk_rag")
def it_helpdesk_rag(question: str) -> dict:
    retrieved = retrieve_it_policy(question, k=2)
    context = "\n\n".join(item["content"] for item in retrieved)

    messages = helpdesk_prompt.format_messages(
        context=context,
        question=question,
    )
    answer = llm.invoke(messages).content

    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
    }

print("✅ Traced IT Helpdesk RAG function is ready")

In [ ]:
# 10.5 Test 1 - Clear Answer Exists in the Knowledge Base
clear_question = "My company laptop was stolen. What should I do, and how quickly should I report it?"
clear_result = it_helpdesk_rag(clear_question)

print("QUESTION:", clear_result["question"])
print("ANSWER:", clear_result["answer"])
print("\nRETRIEVAL:")
for item in clear_result["retrieved"]:
    print(item["metadata"]["policy_id"], "| score:", item["similarity_score"])

In [ ]:
# 10.6 Test 2 - Ambiguous Question
ambiguous_question = "I cannot access company systems from home. What should I check?"
ambiguous_result = it_helpdesk_rag(ambiguous_question)

print("QUESTION:", ambiguous_result["question"])
print("ANSWER:", ambiguous_result["answer"])
print("\nRETRIEVAL:")
for item in ambiguous_result["retrieved"]:
    print(item["metadata"]["policy_id"], "| score:", item["similarity_score"])

In [ ]:
# 10.7 Test 3 - Answer Is Missing from the Knowledge Base
missing_question = "How many paid vacation days do employees receive each year?"
missing_result = it_helpdesk_rag(missing_question)

print("QUESTION:", missing_result["question"])
print("ANSWER:", missing_result["answer"])
print("\nRETRIEVAL:")
for item in missing_result["retrieved"]:
    print(item["metadata"]["policy_id"], "| score:", item["similarity_score"])

## 10.8 What to Inspect in LangSmith

Open the three `it_helpdesk_rag` traces and compare:

- **Retriever output:** Which policies were retrieved for each question?
- **Similarity scores:** Did the retrieved policies actually look relevant?
- **LLM input:** What context was passed to the model?
- **LLM output:** Did the answer stay grounded in that context?
- **Latency:** Which step took the most time?
- **Missing-knowledge case:** Did the model correctly refuse to invent a vacation policy?

### Why this experiment matters
A final answer only shows the result. The trace lets you separate **retrieval behavior** from **generation behavior**, which is the foundation for debugging and evaluating RAG systems.